In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from NOTEBOOKS.calculateEVS import *
from MODELS.pipeline import *
from MODELS.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  # Pass as tuple to calculateEVS
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 1.25


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s25= pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
df = pd.concat([s25, s26])

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

C:\Users\alexg\AppData\Local\Temp\ipykernel_17424\769474497.py:6: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


### Update projected starting lineups

In [5]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\MODELS/teamInfo.py
Updated 18 teams with confirmed lineups


### Top EVs for single bets

In [5]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['ODDS'] <= 250) & (usData['ODDS'] >= -250)]

results = calculateSingleBets(df, singlePTSBookies, model, features, current_date, 
                             edge_threshold=0.20, stake=10, 
                             variance_inflation=1.1, distribution_type='t', 
                             use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)  

singleBets = results
singleBets = singleBets[(singleBets['SIGMA FLAG'] == 'Med') | (singleBets['SIGMA FLAG'] == 'Low')].sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'EXPECTED ROI', 'KELLY_FRACTION','SIGMA FLAG']]
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.sample(5)

Processing single bets with single model...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,EXPECTED ROI,KELLY_FRACTION,SIGMA FLAG
73,Rudy Gobert,BetRivers,10.5,12.56,Over,-137,0,1.53,15.3,0.210,Med
366,Rudy Gobert,Bovada,8.5,12.56,Under,200,0,-4.07,-40.7,0.000,Med
20,Jalen Duren,BetRivers,14.5,12.25,Under,-112,0,3.38,33.8,0.378,Med
156,Aaron Gordon,BetRivers,14.5,15.41,Over,-136,0,0.11,1.1,0.015,Low
277,Reed Sheppard,FanDuel,8.5,8.05,Over,-130,0,-1.99,-19.9,0.000,Med


## Top EVs for 2 leg bets

### Underdog picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=10, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogPairs = results
underdogPairs = underdogPairs[
    underdogPairs[['SIGMA FLAG 1', 'SIGMA FLAG 2']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$','EXPECTED ROI', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,EXPECTED ROI,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Bones Hyland,Deandre Ayton,4.5,17.5,7.67,13.91,over,under,0,5.55,55.5,0.277,Med,Med
1,Sam Hauser,Deandre Ayton,9.5,17.5,6.97,13.91,under,under,0,4.80,48.0,0.240,Med,Med
2,Bones Hyland,Sam Hauser,4.5,9.5,7.67,6.97,over,under,0,4.19,41.9,0.210,Med,Med
3,Terrence Shannon Jr.,Deandre Ayton,5.5,17.5,7.16,13.91,over,under,0,3.66,36.6,0.183,Low,Med
4,Cole Anthony,Deandre Ayton,8.5,17.5,10.39,13.91,over,under,0,3.56,35.6,0.178,Med,Med


### Prizepicks picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=10, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)
pairsPrizepicks = results
pairsPrizepicks = pairsPrizepicks[
    pairsPrizepicks[['SIGMA FLAG 1', 'SIGMA FLAG 2']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'EXPECTED ROI', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,EXPECTED ROI,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Bones Hyland,Deandre Ayton,4.5,17.5,7.67,13.91,over,under,0,55.45,55.5,0.277,Med,Med
1,Sam Hauser,Deandre Ayton,9.5,17.5,6.97,13.91,under,under,0,48.04,48.0,0.240,Med,Med
2,Jalen Duren,Deandre Ayton,14.5,17.5,12.25,13.91,under,under,0,47.88,47.9,0.239,Med,Med
3,Bones Hyland,Sam Hauser,4.5,9.5,7.67,6.97,over,under,0,41.91,41.9,0.210,Med,Med
4,Bones Hyland,Jalen Duren,4.5,14.5,7.67,12.25,over,under,0,41.75,41.8,0.209,Med,Med


## 3 leg parlay

### Underdog picks

In [12]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.10, stake=100, 
                     variance_inflation=1.1, distribution_type='t', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogTrios = threeLeg
underdogTrios = threeLeg[
    threeLeg[['SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'EXPECTED ROI', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,EXPECTED ROI,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Bones Hyland,Sam Hauser,Deandre Ayton,4.5,9.5,17.5,7.67,6.97,13.91,over,under,under,0,97.97,98.0,0.196,Med,Med,Med
1,Terrence Shannon Jr.,Bones Hyland,Deandre Ayton,5.5,4.5,17.5,7.16,7.67,13.91,over,over,under,0,82.66,82.7,0.165,Low,Med,Med
2,Cole Anthony,Bones Hyland,Deandre Ayton,8.5,4.5,17.5,10.39,7.67,13.91,over,over,under,0,81.29,81.3,0.163,Med,Med,Med
3,Jaylen Clark,Bones Hyland,Deandre Ayton,5.5,4.5,17.5,7.14,7.67,13.91,over,over,under,0,80.96,81.0,0.162,Med,Med,Med
4,Bones Hyland,Cedric Coward,Deandre Ayton,4.5,12.5,17.5,7.67,10.96,13.91,over,under,under,0,79.70,79.7,0.159,Med,Med,Med


### Prizepicks picks

In [13]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.40, stake=100, 
                     variance_inflation=1.1, distribution_type='normal', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

triosPrizepicks = threeLeg  
triosPrizepicks = threeLeg[
    threeLeg[['SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3','MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'EXPECTED ROI', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,EXPECTED ROI,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Bones Hyland,Sam Hauser,Deandre Ayton,4.5,9.5,17.5,7.67,6.97,13.91,over,under,under,0,70.05,70.0,0.140,Med,Med,Med
1,Bones Hyland,Jalen Duren,Deandre Ayton,4.5,14.5,17.5,7.67,12.25,13.91,over,under,under,0,69.69,69.7,0.139,Med,Med,Med
2,Bones Hyland,Jaylen Wells,Deandre Ayton,4.5,10.0,17.5,7.67,11.84,13.91,over,over,under,0,63.24,63.2,0.126,Med,Med,Med
3,Terrence Shannon Jr.,Bones Hyland,Deandre Ayton,5.5,4.5,17.5,7.16,7.67,13.91,over,over,under,0,61.40,61.4,0.123,Low,Med,Med
4,Sam Hauser,Jalen Duren,Deandre Ayton,9.5,14.5,17.5,6.97,12.25,13.91,under,under,under,0,60.76,60.8,0.122,Med,Med,Med


In [14]:
def count_line_hits(player_df, line, category, game_windows=[5, 10, 15]):
    results = {}
    player_df_sorted = player_df.sort_values('GAME_DATE')
    total_games = len(player_df_sorted)

    for window in game_windows:
        # Handle players with fewer games
        if total_games < window:
            last_n_games = player_df_sorted
        else:
            last_n_games = player_df_sorted.tail(window)

        if category == 'player_points':
            hits = (last_n_games['PTS'] > line).sum()
        elif category == 'player_assists':
            hits = (last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds':
            hits = (last_n_games['REB'] > line).sum()
        elif category == 'player_threes':
            hits = (last_n_games['FG3M'] > line).sum()
        elif category == 'player_blocks':
            hits = (last_n_games['BLK'] > line).sum()
        elif category == 'player_steals':
            hits = (last_n_games['STL'] > line).sum()
        elif category == 'player_field_goals':
            hits = (last_n_games['FGM'] > line).sum()
        elif category == 'player_frees_made':
            hits = (last_n_games['FTM'] > line).sum()
        elif category == 'player_points_rebounds_assists':
            hits = (last_n_games['PTS'] + last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_points_rebounds':
            hits = (last_n_games['PTS'] + last_n_games['REB'] > line).sum()
        elif category == 'player_points_assists':
            hits = (last_n_games['PTS'] + last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds_assists':
            hits = (last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_turnovers':
            hits = (last_n_games['TOV'] > line).sum()
        else:
            hits = 0

        results['NAME'] = player_df_sorted['PLAYER_NAME'].iloc[0] if total_games > 0 else 'Unknown'
        results['CATEGORY'] = category
        results['LINE'] = line
        results[f'L-{window}'] = round(hits / window, 2)


    return results

prizePicks = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks')]
prizePicks= prizePicks.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
prizePicks

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
178,PrizePicks,player_points,Aaron Nesmith,Over,16.0,-137,2025-11-04,2025-11-03T23:07:47Z
180,PrizePicks,player_points,Giannis Antetokounmpo,Over,31.5,-137,2025-11-04,2025-11-03T23:07:47Z
182,PrizePicks,player_points,Pascal Siakam,Over,26.5,-137,2025-11-04,2025-11-03T23:07:47Z
184,PrizePicks,player_points,Jarace Walker,Over,14.5,-137,2025-11-04,2025-11-03T23:07:47Z
186,PrizePicks,player_points,Myles Turner,Over,14.5,-137,2025-11-04,2025-11-03T23:07:47Z
...,...,...,...,...,...,...,...,...
3269,PrizePicks,player_blocks_steals,Tim Hardaway Jr,Over,0.5,-137,2025-11-04,2025-11-03T23:08:00Z
3271,PrizePicks,player_blocks_steals,Jonas Valanciunas,Over,0.5,-137,2025-11-04,2025-11-03T23:08:00Z
3273,PrizePicks,player_blocks_steals,Jrue Holiday,Over,1.5,-137,2025-11-04,2025-11-03T23:07:36Z
3275,PrizePicks,player_blocks_steals,Kris Murray,Over,1.5,-137,2025-11-04,2025-11-03T23:07:36Z


In [15]:
line_hit_data = []

for index, row in prizePicks.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

        
        

Saved 123 records for player_points to player_points.csv
Saved 72 records for player_rebounds to player_rebounds.csv
Saved 44 records for player_assists to player_assists.csv
Saved 22 records for player_threes to player_threes.csv
Saved 6 records for player_blocks to player_blocks.csv
Saved 19 records for player_steals to player_steals.csv
Saved 38 records for player_field_goals to player_field_goals.csv
Saved 24 records for player_frees_made to player_frees_made.csv
Saved 16 records for player_frees_attempts to player_frees_attempts.csv
Saved 120 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 105 records for player_points_rebounds to player_points_rebounds.csv
Saved 106 records for player_points_assists to player_points_assists.csv
Saved 69 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 15 records for player_turnovers to player_turnovers.csv
Saved 19 records for player_blocks_steals to player_blocks_steals.csv

All cate

In [16]:
underdog = dfsData[(dfsData['BOOKMAKER'] == 'Underdog')]
underdog= underdog.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
line_hit_data = []

for index, row in underdog.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

Saved 79 records for player_points to player_points.csv
Saved 29 records for player_rebounds to player_rebounds.csv
Saved 22 records for player_assists to player_assists.csv
Saved 22 records for player_threes to player_threes.csv
Saved 1 records for player_steals to player_steals.csv
Saved 3 records for player_frees_made to player_frees_made.csv
Saved 92 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 36 records for player_points_rebounds to player_points_rebounds.csv
Saved 36 records for player_points_assists to player_points_assists.csv
Saved 18 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 8 records for player_turnovers to player_turnovers.csv
Saved 3 records for player_blocks_steals to player_blocks_steals.csv

All category files saved to ../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG
